# Train Qwen2.5-0.5B — Clean-Only Ablation

This notebook is a **controlled ablation** of `train_qwen05.py`.  
The **only** change from the full training run is the dataset:  
- Full run: `training_data_augmented_v2.jsonl` — **1487 examples** (clean + backtracking)  
- This run: `training_data_clean_only.jsonl` — **446 examples** (clean only)  

All hyperparameters are identical.

---
**Prerequisites**
1. Run `make_clean_only_dataset.ipynb` first to generate `training_data_clean_only.jsonl`
2. Install dependencies: `pip install transformers datasets torch accelerate`
3. Recommended: GPU with ≥8 GB VRAM (e.g. T4, RTX 3080, A100)

## Install dependencies (run once)

In [ ]:
# Uncomment and run if not already installed
# !pip install transformers datasets torch accelerate

## Configuration
Only `JSONL_PATH` and `SAVE_PATH` differ from the full training run.

In [25]:
# ── Paths (the ONLY two differences from train_qwen05.py) ────────────────────
JSONL_PATH = "Backtrack_Count(Ablation)\\training_data_balanced_equal.jsonl"   # 446 clean examples
SAVE_PATH  = "./qwen05_balanced_equal_finetuned"    # where the model is saved

# ── Model ─────────────────────────────────────────────────────────────────────
MODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"

# ── Hyperparameters (identical to train_qwen05.py) ────────────────────────────
SEED         = 42
MAX_LENGTH   = 1024
LR           = 1.5e-5
WEIGHT_DECAY = 0.01
BATCH_SIZE   = 4
GRAD_ACCUM   = 4       # effective batch = 16
SCHEDULER    = "cosine"
NUM_EPOCHS   = 5
WARMUP_RATIO = 0.06

## Prompt templates

In [26]:
SYSTEM_PROMPT = (
    "Use numbers and basic arithmetic operations (+, -, *, /) to obtain 24. "
    "Each step, you are only allowed to choose two of the remaining numbers to obtain a new number.\n"
    "Step 1: Start by considering possible operations for each pair of numbers.\n"
    "Step 2: Try a path (a pair of two numbers), see if the remaining numbers can possibly reach the goal 24. If not, backtrack and attempt another.\n"
    "Step 3: Branch out to try different orders of operations and combinations, evaluating each outcome.\n"
    "Step 4: If one path doesn't lead to a solution, backtrack and try alternative operations.\n\n"
)

IN_CONTEXT_HEADER = (
    "Here are some solved examples:\n\n"
    "Numbers: 4 4 6 8. Target: 24.\n"
    "Use each number exactly once with +, -, *, / to reach 24.\n"
    "Steps:\n"
    "4 + 8 = 12 (left: 4 6 12)\n"
    "6 - 4 = 2 (left: 2 12)\n"
    "2 * 12 = 24 (left: 24)\n"
    "Answer: (6 - 4) * (4 + 8) = 24\n\n"
    "Numbers: 1 4 8 8. Target: 24.\n"
    "Use each number exactly once with +, -, *, / to reach 24.\n"
    "Steps:\n"
    "8 / 4 = 2 (left: 1 2 8)\n"
    "1 + 2 = 3 (left: 3 8)\n"
    "3 * 8 = 24 (left: 24)\n"
    "Answer: (1 + 8 / 4) * 8 = 24\n\n"
    "Numbers: 5 5 5 9. Target: 24.\n"
    "Use each number exactly once with +, -, *, / to reach 24.\n"
    "Steps:\n"
    "5 + 5 = 10 (left: 5 9 10)\n"
    "10 + 5 = 15 (left: 9 15)\n"
    "15 + 9 = 24 (left: 24)\n"
    "Answer: ((5 + 5) + 5) + 9 = 24\n\n"
    "Numbers: 4 9 10 13. Target: 24.\n"
    "Use each number exactly once with +, -, *, / to reach 24.\n"
    "Steps:\n"
    "4 + 9 = 13 (left: 10 13 13)\n"
    "13 - 10 = 3 (left: 3 13)\n"
    "13 + 3 = 16 (left: 16)\n"
    "Backtrack. (to: 4 9 10 13).\n"
    "13 - 10 = 3 (left: 3 4 9)\n"
    "9 - 3 = 6 (left: 4 6)\n"
    "4 * 6 = 24 (left: 24)\n"
    "Answer: 4 * (9 - (13 - 10)) = 24\n\n"
    "Now solve this puzzle:\n"
)

## Load and split dataset

In [27]:
import json
import random
from datasets import Dataset

print("Loading dataset...")
with open(JSONL_PATH) as f:
    examples = [json.loads(line) for line in f if line.strip()]

random.seed(SEED)
unique_puzzles = list({e["puzzle"] for e in examples})
random.shuffle(unique_puzzles)
split_idx     = int(len(unique_puzzles) * 0.9)
train_puzzles = set(unique_puzzles[:split_idx])

train_rows = [e for e in examples if e["puzzle"] in train_puzzles]
val_rows   = [e for e in examples if e["puzzle"] not in train_puzzles]

# Zero-overlap assertion
assert set(e["puzzle"] for e in train_rows).isdisjoint(
       set(e["puzzle"] for e in val_rows)), "Puzzle leakage!"

print(f"Train: {len(train_rows)} rows | Val: {len(val_rows)} rows")
print("Zero train/val puzzle overlap confirmed ✓")

eff_batch       = BATCH_SIZE * GRAD_ACCUM
steps_per_epoch = len(train_rows) // eff_batch
total_steps     = steps_per_epoch * NUM_EPOCHS
print(f"Eff batch: {eff_batch} | Steps/epoch: {steps_per_epoch} | Total steps: {total_steps}")

train_dataset = Dataset.from_list(train_rows)
val_dataset   = Dataset.from_list(val_rows)

Loading dataset...
Train: 1339 rows | Val: 148 rows
Zero train/val puzzle overlap confirmed ✓
Eff batch: 16 | Steps/epoch: 83 | Total steps: 415


## Load tokenizer

In [28]:
from transformers import AutoTokenizer

print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

if tokenizer.pad_token is None or tokenizer.pad_token == tokenizer.eos_token:
    tokenizer.add_special_tokens({"pad_token": "<|pad|>"})

tokenizer.padding_side = "right"
print("Tokenizer loaded ✓")

Loading tokenizer...
Tokenizer loaded ✓


## Tokenize dataset

In [29]:
def preprocess(batch):
    all_input_ids = []
    all_attention  = []
    all_labels     = []

    for prompt_field, completion in zip(batch["prompt"], batch["completion"]):
        user_content = SYSTEM_PROMPT + IN_CONTEXT_HEADER + prompt_field.split("Now solve this puzzle:\n")[-1].strip()

        prompt_messages = [{"role": "user", "content": user_content}]
        prompt_text = tokenizer.apply_chat_template(
            prompt_messages, tokenize=False, add_generation_prompt=True
        )
        prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]

        completion_text = completion + tokenizer.eos_token
        completion_ids  = tokenizer(completion_text, add_special_tokens=False)["input_ids"]

        max_completion = MAX_LENGTH - len(prompt_ids)
        if max_completion <= 0:
            continue
        completion_ids = completion_ids[:max_completion]

        full_ids = prompt_ids + completion_ids
        pad_len  = MAX_LENGTH - len(full_ids)

        input_ids = full_ids + [tokenizer.pad_token_id] * pad_len
        attention  = [1] * len(full_ids) + [0] * pad_len
        labels     = [-100] * len(prompt_ids) + completion_ids + [-100] * pad_len

        assert len(input_ids) == MAX_LENGTH
        assert len(labels)    == MAX_LENGTH

        all_input_ids.append(input_ids)
        all_attention.append(attention)
        all_labels.append(labels)

    return {"input_ids": all_input_ids, "attention_mask": all_attention, "labels": all_labels}


print("Tokenizing...")
tok_train = train_dataset.map(preprocess, batched=True, remove_columns=train_dataset.column_names)
tok_val   = val_dataset.map(preprocess,   batched=True, remove_columns=val_dataset.column_names)
print(f"  {len(tok_train)} train | {len(tok_val)} val after tokenization")

Tokenizing...


Map: 100%|██████████| 148/148 [00:00<00:00, 678.62 examples/s]

  1339 train | 148 val after tokenization


## Sanity check

In [30]:
print("Sanity check — completion labels:")
for idx in range(min(3, len(tok_train))):
    ex         = tok_train[idx]
    label_toks = [l for l in ex["labels"] if l != -100]
    decoded    = tokenizer.decode(label_toks[:40])
    try:
        first_comp = next(i for i, l in enumerate(ex["labels"]) if l != -100)
    except StopIteration:
        first_comp = len(ex["labels"])

    print(f"  [{idx}] prompt={first_comp} tok | completion={len(label_toks)} tok | "
          f"total={first_comp + len(label_toks)}/{MAX_LENGTH}")
    print(f"        first 40 decoded: {decoded!r}")

    if len(label_toks) == 0:
        raise ValueError(f"Example {idx} has NO completion tokens.")
    if first_comp > MAX_LENGTH - 20:
        raise ValueError(f"Example {idx}: prompt leaves < 20 tokens for completion.")

print("  Sanity check passed ✓")

Sanity check — completion labels:
  [0] prompt=703 tok | completion=69 tok | total=772/1024
        first 40 decoded: '1 + 1 = 2 (left: 2 4 12)\n12 / 2 = 6 (left: 6 4)\n6 * 4 = '
  [1] prompt=703 tok | completion=194 tok | total=897/1024
        first 40 decoded: '13 + 2 = 15 (left: 15 1 1)\n1 - 15 = -14 (left: -14 1)\n-1'
  [2] prompt=704 tok | completion=75 tok | total=779/1024
        first 40 decoded: '13 + 7 = 20 (left: 20 3 12)\n20 - 12 = 8 (left: 8 3)\n8 *'
  Sanity check passed ✓


## Load model

In [31]:
import torch
from transformers import AutoModelForCausalLM

print("Loading model...")
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    device_map="auto",
    torch_dtype=torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16,
)

if len(tokenizer) != model.config.vocab_size:
    model.resize_token_embeddings(len(tokenizer))

model.gradient_checkpointing_enable()
print(f"Model loaded on {next(model.parameters()).device} ✓")

Loading model...


Loading weights: 100%|██████████| 290/290 [00:00<00:00, 334.76it/s]


Model loaded on cuda:0 ✓


## Train

In [32]:
from transformers import Trainer, TrainingArguments, EarlyStoppingCallback

EVAL_SAVE_STEPS = max(10, steps_per_epoch)

training_args = TrainingArguments(
    output_dir=SAVE_PATH,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=NUM_EPOCHS,
    learning_rate=LR,
    weight_decay=WEIGHT_DECAY,
    lr_scheduler_type=SCHEDULER,
    warmup_ratio=WARMUP_RATIO,
    eval_strategy="steps",
    eval_steps=EVAL_SAVE_STEPS,
    save_strategy="steps",
    save_steps=EVAL_SAVE_STEPS,
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=3,
    logging_steps=5,
    logging_dir="./train_logs_qwen05_clean_only",
    bf16=torch.cuda.is_bf16_supported(),
    fp16=not torch.cuda.is_bf16_supported(),
    push_to_hub=False,
    remove_unused_columns=False,
    report_to="none",
    seed=SEED,
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tok_train,
    eval_dataset=tok_val,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("=" * 60)
print(f"Model    : {MODEL_NAME}")
print(f"Data     : {JSONL_PATH} ({len(train_rows)} train examples)")
print(f"Epochs   : {NUM_EPOCHS} | Eff batch: {eff_batch} | LR: {LR:.1e}")
print(f"Scheduler: {SCHEDULER} | Total steps: ~{total_steps}")
print("=" * 60)

trainer.train()

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] `logging_dir` is deprecated and will be removed in v5.2. Please set `TENSORBOARD_LOGGING_DIR` instead.


Model    : Qwen/Qwen2.5-0.5B-Instruct
Data     : Backtrack_Count(Ablation)\training_data_balanced_equal.jsonl (1339 train examples)
Epochs   : 5 | Eff batch: 16 | LR: 1.5e-05
Scheduler: cosine | Total steps: ~415


Step,Training Loss,Validation Loss
83,0.097074,0.116690
166,0.083827,0.102090
249,0.070329,0.102905
332,0.067289,0.105332


Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.35s/it]
[transformers] There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=332, training_loss=0.09944974500910346, metrics={'train_runtime': 654.6372, 'train_samples_per_second': 10.227, 'train_steps_per_second': 0.642, 'total_flos': 1.1647711022678016e+16, 'train_loss': 0.09944974500910346, 'epoch': 3.955223880597015})

## Save model and logs

In [33]:
trainer.save_model(SAVE_PATH)
tokenizer.save_pretrained(SAVE_PATH)

with open("training_logs_qwen05_balanced_equal.json", "w") as f:
    json.dump(trainer.state.log_history, f, indent=2)

final_eval = trainer.state.log_history[-1]
print(f"Done. Model saved -> {SAVE_PATH}")
print(f"Logs   -> training_logs_qwen05_balanced_equal.json")
print(f"Final eval loss: {final_eval.get('eval_loss', 'see logs')}")

Writing model shards: 100%|██████████| 1/1 [00:03<00:00,  3.21s/it]

Done. Model saved -> ./qwen05_balanced_equal_finetuned
Logs   -> training_logs_qwen05_balanced_equal.json
Final eval loss: see logs
